In [ ]:
import pandas as pd
import requests
import json
import time
import nltk
import spacy

from nltk.corpus import framenet as fn
from nltk.corpus.reader.framenet import PrettyList
from nltk.tokenize import sent_tokenize
from openai import OpenAI
from tqdm import tqdm

In [ ]:
frames = list()
for frame in PrettyList(fn.frames()):
    frame_str = str()
    frame_str += f"{frame.name}"
    # frame_str += f"Frame:\n{frame.name}"
    # frame_str += f"\nDefinition:\n{frame.definition}"
    # frame_str += f"\nFrame Elements:\n"
    # num_fes = 1
    # for fe in frame.FE.keys():
    #     frame_str += f"{num_fes}. {fe}\n"
    #     num_fes += 1
    frame_str += '\n'
    frames.append(frame_str)

len(frames)

In [ ]:
for frame in PrettyList(fn.frames()):
    print(frame.frameRelations)
    break

In [ ]:
print(''.join(frames))

In [ ]:
reddit_df = pd.read_csv("../../corpora/es_reddit/processed_es_reddit.csv", header=None)
reddit = reddit_df[0].tolist()

reddit = reddit[:40]

In [ ]:
news_df = pd.read_csv("../../corpora/es_spanish_news/processed_spanish_news.csv", header=None)
news = news_df[0].tolist()

news = news[:4]

In [ ]:
with open("../../corpora/es_cereal/processed_cereal_es.txt", 'r') as f:
    cereal = f.readlines()

cereal = cereal[:2]

In [ ]:
cereal = reddit + news + cereal

len(cereal)

In [ ]:
tokenizer = nltk.data.load('tokenizers/punkt/spanish.pickle')

In [ ]:
nlp = spacy.load("es_core_news_sm")

In [ ]:
def split_into_clauses_spacy(sentence):
    """Splits a sentence into clauses using spaCy's dependency parser."""
    doc = nlp(sentence)
    clauses = []
    start = 0
    for i, token in enumerate(doc):
        # Detect clause boundaries based on syntactic relations (e.g., conjunctions, subordinating conjunctions)
        if token.dep_ == "cc" or token.dep_ == "mark": # cc: coordinating conjunction, mark: marker (subordinating conjunction)
            clauses.append(doc[start:token.i].text)
            start = token.i
    clauses.append(doc[start:].text)
    return [clause.strip() for clause in clauses if clause.strip()]

In [ ]:
example = """Para el texto: "Ella corre rápido"."

Las palabras y anotaciones podrían ser:

[
  {"word": "Ella", "fee": false, "frame": "-"},
  {"word": "corre", "fee": true, "frame": "Self_motion"},
  {"word": "rápido", "fee": true, "frame": "Speed_description"}
]
"""

In [ ]:
PROMPT = """TAREA: Anote el TEXTO dado palabra por palabra con frames de Berkeley FrameNet de la lista FRAMES proporcionada.

INSTRUCCIONES:
1. Utilizando la teoría de la semántica de frames de Fillmore, inspeccione cada palabra del TEXTO considerando su contexto.
2. Para cada palabra, asigne los parámetros en formato de matriz JSON: 'word', 'fee' y 'frame'.
3. Establezca 'fee' como TRUE si la palabra evoca o activa un frame de FrameNet (por ejemplo, verbos que denotan acciones, sustantivos que denotan entidades o conceptos relacionados con frames). De lo contrario, establezca 'fee' como FALSE.
4. Si 'fee' es TRUE, entonces para el parámetro 'frame' seleccione un frame apropiado de FRAMES o establézcalo como 'To_Review', lo que significa que no hay frames adecuados en FRAMES. Solo puedes usar frames de la lista FRAMES.
5. Si la palabra es un adverbio, busque un frame de FrameNet existente que evoca y asígnelo. Si no hay frames apropiados en la lista FRAMES, configure "frame" como "To_Review".
6. Si la palabra es un nombre propio, establezca 'fee' en FALSE.
7. Si la palabra es un verbo auxiliar (por ejemplo, soy, es, son, fue, fueron), entonces establezca 'fee' en FALSE.
8. Si 'fee' es FALSE, establezca 'frame' como '-'.
9. Proporcione un único objeto de matriz JSON con todas las claves 'word', 'fee' y 'frame' y sus respectivos valores.
10. Imprima únicamente el objeto de matriz JSON, sin explicaciones ni comentarios.
11. No revise ni genere múltiples respuestas.

EJEMPLO:
{example}

TEXTO: {text}

FRAMES: {bfn_frames}
"""

In [ ]:
client = OpenAI(
  api_key=""
)

In [ ]:
responses = list()
all_sentences = list()
for text in tqdm(cereal, position=0):
    sentences = tokenizer.tokenize(text)
    for sent in sentences:
        subsentences = split_into_clauses_spacy(sent)
        for subsent in subsentences:
            all_sentences.append(subsent)

            text_prompt = PROMPT.format(example=example,
                                        text=subsent,
                                        bfn_frames=''.join(frames))
            completion = client.chat.completions.create(
                model="gpt-4.1-mini-2025-04-14",
                store=True,
                messages=[
                    {
                        "role": "user",
                        "content": text_prompt
                    }
                ]
            )

            response = completion.choices[0].message.content
            responses.append(response)
            
            # break

In [ ]:
print(responses[0])

In [ ]:
len(all_sentences)

In [ ]:
len(responses)

In [ ]:
df_dict = {"sents": all_sentences, "preds": responses}
df = pd.DataFrame(df_dict)

df

In [ ]:
df.to_csv("../../outputs/fee_annotation/es_cereal_46_gpt_preds_subsentences.csv", index=False)